# 诗歌生成

# 数据处理

In [11]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
#重复导入了 from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

#全局常量--标记序列的起始和结束
start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r',encoding='utf-8') as fd:#补充个以utf-8编码形式打开的
        for line in fd:
            outs = line.strip().split(':')#通过strip()去除首尾空白，然后按 : 分割成列表 outs。
            content = ''.join(outs[1:])#将 : 之后的部分拼接成完整的诗歌字符串。
            ins = [start_token] + list(content) + [end_token] 
            #如果序列长度超过 200，则跳过（过滤掉过长的诗）
            if len(ins) > 200:
                continue
            examples.append(ins)
    #collections.Counter 统计所有字符（包括特殊标记）的出现频次。        
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    #最终 words 是 ('PAD','UNK') 加上所有频次降序的字符。
    words = ('PAD', 'UNK') + words[:len(words)]
    #建立词到索引的映射字典。
    word2id = dict(zip(words, range(len(words))))
    #建立索引到词的逆映射字典。
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('./poems.txt')#处理 ./poems.txt 文件，得到数据实例、词汇映射。[vscode打开成utf-8编码了]
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

"""
从文本文件读取诗歌，每行格式 标题:内容。
为每首诗添加 bos 和 eos 标记，并过滤掉过长诗歌。
统计所有字符，建立包含 PAD、UNK 的词汇表，并按频次排序分配索引。
将诗歌转换为索引序列，并与长度配对。
使用 TensorFlow 数据集 API 将数据转换为批次，并进行填充。
将每个批次转换为输入（前 n-1 个 token）和标签（后 n-1 个 token），用于训练自回归语言模型。
返回数据集及词汇表，可供后续模型训练使用。
"""

'\n从文本文件读取诗歌，每行格式 标题:内容。\n为每首诗添加 bos 和 eos 标记，并过滤掉过长诗歌。\n统计所有字符，建立包含 PAD、UNK 的词汇表，并按频次排序分配索引。\n将诗歌转换为索引序列，并与长度配对。\n使用 TensorFlow 数据集 API 将数据转换为批次，并进行填充。\n将每个批次转换为输入（前 n-1 个 token）和标签（后 n-1 个 token），用于训练自回归语言模型。\n返回数据集及词汇表，可供后续模型训练使用。\n'

# 模型代码， 完成建模代码

In [ ]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)  #词汇表大小

        #Keras3中Embedding不再接受 batch_input_shape 参数
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        #隐藏层，维度128
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        # 将rnncell变成一个可以处理序列的层
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        # 全连接层，将每个时间步的隐藏状态映射为词汇表大小的 logits
        self.dense = tf.keras.layers.Dense(self.v_sz)

    def call(self, inp_ids, training=False):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        emb = self.embed_layer(inp_ids)          # (batch, seq_len, 64)
        rnn_out = self.rnn_layer(emb, training=training)  # (batch, seq_len, 128)
        logits = self.dense(rnn_out)             # (batch, seq_len, vocab_size)
        return logits

    @tf.function
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,]
        '''
        inp_emb = self.embed_layer(x)  # shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, state)  # shape(b_sz, h_sz)
        logits = self.dense(h)  # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [4]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [ ]:
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = compute_loss(logits, y, seqlen)

    grads = tape.gradient(loss, model.trainable_variables)
    grads_and_vars = [(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None]
    if not grads_and_vars:
        raise ValueError('No gradients provided.')
    optimizer.apply_gradients(grads_and_vars)
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', float(loss.numpy()))

    return loss

# 训练优化过程

In [17]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

#先做一次前向，确保变量创建并被优化器追踪
_dummy_x = tf.zeros((1, 5), dtype=tf.int64)
_ = model(_dummy_x, training=False)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 8.821582794189453
epoch 1 : loss 6.552888870239258
epoch 2 : loss 6.047905921936035
epoch 3 : loss 5.821629047393799
epoch 4 : loss 5.612483978271484
epoch 5 : loss 5.43842887878418
epoch 6 : loss 5.382233142852783
epoch 7 : loss 5.351621150970459
epoch 8 : loss 5.270983695983887
epoch 9 : loss 5.201240062713623


# 生成过程

In [19]:
def gen_sentence():
    state = [tf.random.normal(shape=(1, 128), stddev=0.5), tf.random.normal(shape=(1, 128), stddev=0.5)]
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        collect.append(cur_token.numpy()[0])
    return [id2word[t] for t in collect]
print(''.join(gen_sentence()))

南山一片，不知何事。eos得不知君，不知何所见。eos生不可见，不得无人间。eos子不可见，不知何所知。eos生不可


In [18]:
#生成以“日 、 红 、 山 、 夜 、 湖、 海 、 月”开头的诗歌
def gen_sentence_with_start(start_word, max_len=50):
    if start_word not in word2id:
        #开头字不在词表中，使用UNK
        start_id = word2id.get('UNK', 1)
        start_word = id2word[start_id]
    else:
        start_id = word2id[start_word]

    #SimpleRNNCell只有1个state
    state = [tf.zeros((1, 128), dtype=tf.float32)]

    bos_id = word2id['bos']
    eos_id = word2id['eos']

    #先喂bos
    cur_token = tf.constant([bos_id], dtype=tf.int32)
    _, state = model.get_next_token(cur_token, state)

    #指定首字
    cur_token = tf.constant([start_id], dtype=tf.int32)
    result = [start_word]

    #继续生成后续字
    for _ in range(max_len - 1):
        next_token, state = model.get_next_token(cur_token, state)
        tid = int(next_token.numpy()[0])

        if tid == eos_id:
            break

        result.append(id2word.get(tid, ''))
        cur_token = next_token

    return ''.join(result)

for w in ['日', '红', '山', '夜', '湖', '海', '月']:
    print(w, '->', gen_sentence_with_start(w, max_len=50))

日 -> 日日无人，不知何处，不知何处。
红 -> 红叶满山山，风风吹雨声。
山 -> 山上春风起，山中不可知。
夜 -> 夜雨无人，一枝花下。
湖 -> 湖上春风起，山中不可知。
海 -> 海上春风起，山中不可知。
月 -> 月落花中日，风吹白水中。
